# Ngày 19 · PCA bằng SVD

- Tính PCA từ SVD của dữ liệu đã trừ trung bình
- Chiếu xuống k chiều, dựng lại và đo sai số dựng lại
- Chọn k theo tỉ lệ phương sai giữ lại

**Đọc:**
- ISLR (bản 2), §12.2 Principal Components Analysis
- Ng & Ma, CS229 Lecture Notes (2023), Chương 12 Principal components analysis
- Hastie, Tibshirani & Friedman, ESL, §14.5.1 Principal Components

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
from sklearn.datasets import load_digits
# Optical Recognition of Handwritten Digits (UCI), có sẵn trong scikit-learn: 1797 ảnh 8×8, giá trị 0–16.
X_dg, y_dg = load_digits(return_X_y=True)
from sklearn.decomposition import PCA
_sk19 = PCA(10).fit(X_dg)
def _fit_pca_ref(X, k):
    mean = X.mean(0); _, S, Vt = np.linalg.svd(X - mean, full_matrices=False); ev = S ** 2 / (len(X) - 1)
    return {'mean': mean, 'components': Vt[:k], 'explained_variance': ev[:k], 'explained_variance_ratio': ev[:k] / ev.sum()}


## fit, transform và dựng lại

Gợi ý: np.linalg.svd(X − mean, full_matrices=False). Hàng của Vt là các thành phần. Phương sai giải thích = S²/(n − 1). Dựng lại: Z @ components + mean.

In [ ]:
def fit_pca(X, k):
    # Trả dict: mean, components (k, d), explained_variance (k,), explained_variance_ratio (k,).
    raise NotImplementedError

def transform(model, X):
    raise NotImplementedError

def inverse_transform(model, Z):
    raise NotImplementedError


In [ ]:
_c19 = {}
def _m19():
    if 'm' not in _c19:
        _c19['m'] = fit_pca(X_dg, 10)
    return _c19['m']
_report('d19-pca',
    _check('Thành phần khớp scikit-learn (không xét dấu)', lambda: (np.allclose(np.abs((_m19()['components'] * _sk19.components_).sum(1)), 1, atol=1e-8), '10 thành phần trên load_digits.')),
    _check('Phương sai và tỉ lệ phương sai khớp', lambda: (np.allclose(_m19()['explained_variance'], _sk19.explained_variance_) and np.allclose(_m19()['explained_variance_ratio'], _sk19.explained_variance_ratio_), 'Chia cho n − 1.')),
    _check('Sai số dựng lại bằng tổng phương sai bị bỏ', lambda: ((lambda full: np.isclose(((X_dg - inverse_transform(_m19(), transform(_m19(), X_dg))) ** 2).sum(1).mean(), full['explained_variance'][10:].sum() * (len(X_dg) - 1) / len(X_dg)))(_fit_pca_ref(X_dg, 64)), 'Trung bình ‖x − x̂‖² = Σ phương sai các thành phần bị bỏ × (n − 1)/n.')),
)


## Chọn số thành phần theo phương sai giữ lại

Gợi ý: Lấy explained_variance_ratio của mọi thành phần, np.cumsum, rồi tìm vị trí đầu tiên ≥ ratio (np.searchsorted) và cộng 1 vì đếm từ 1.

In [ ]:
def n_components_for(X, ratio):
    # Số thành phần nhỏ nhất có tổng tỉ lệ phương sai >= ratio. Dùng fit_pca của bài trên.
    raise NotImplementedError


In [ ]:
fit_pca = globals().get('fit_pca') or _fit_pca_ref
_report('d19-k',
    _check('Khớp PCA(n_components=ratio) của scikit-learn', lambda: (all(n_components_for(X_dg, r) == PCA(r, svd_solver='full').fit(X_dg).n_components_ for r in (.5, .8, .95)), 'ratio = 0,5; 0,8; 0,95.')),
    _check('Giữ 95% phương sai của load_digits cần 29 thành phần', lambda: (n_components_for(X_dg, .95) == 29, 'Trên 64 pixel.')),
)


## Dự án

Trên ảnh Digit Recognizer 28×28 (hoặc load_digits 8×8), vẽ 16 thành phần chính đầu tiên thành ảnh ("eigen-digit"), vẽ đường tỉ lệ phương sai tích lũy, và dựng lại 5 ảnh với k = 10, 50, 150. Ghi k nhỏ nhất giữ 95% phương sai.